In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ✈ **Tarea: Predicción de la resistencia del concreto mediante modelos lineales**


Una empresa dedicada a la producción de concreto desea mejorar su capacidad para estimar la resistencia de sus productos.

Actualmente, la empresa dispone de información histórica sobre la composición de diferentes mezclas de concreto, incluyendo la cantidad de cemento, agua, agregados, aditivos y el tiempo de curado.

La empresa desea aprovechar estos datos para desarrollar modelos predictivos que permitan estimar la resistencia del concreto en megapascales (MPa).

Para ello, se solicita desarrollar un análisis utilizando las técnicas de regresión lineal estudiadas en clase.

## **Objetivo de la tarea:**

Construir y comparar diferentes modelos de regresión lineal que permitan estimar la resistencia del concreto a partir de las características de su composición.

# **🔵 Parte 1. Carga y exploración de datos.**

    a. Cargar el dataset proporcionado en Python.

    b. Visualizar los primeros registros.

    c. Identificar las dimensiones del dataset.

    d. Verificar si existen valores faltantes.

    e. Obtener estadísticas descriptivas de las variables.

    f. Identificar cuál es la variable dependiente y cuáles son las variables independientes.

**Pregunta de análisis: ¿Qué información contiene el dataset y qué queremos predecir?**

## **DESARROLLO PARTE 1.**

In [ ]:
import pandas as pd
from scipy.stats import kurtosis, skew

###### a. Cargar datos del dataset
df = pd.read_csv('/content/drive/MyDrive/ML Regresion con Python/Dataset/Concrete_Data.csv')

print("=== CHECKLIST DE CALIDAD DE DATOS ===")

###### b. Visualizar los primeros registros.
df.head(10)
print(df.head())

###### c. Identificar las dimensiones del dataset
print("Número de filas (registros):", df.shape[0])
print("Número de columnas (variables):", df.shape[1])

###### d. Verificar si existen valores faltantes.
#-- Conteo de valores faltantes por columna
print(df.isnull().sum())

#-- Verificar si existe al menos un valor faltante en todo el dataset
print("¿Hay valores faltantes?:", df.isnull().values.any())

#-- Porcentaje de valores faltantes por columna (más detallado)
print((df.isnull().sum() / len(df)) * 100)

###### e. Obtener estadísticas descriptivas de las variables.
# variables numéricas
print(df.describe())

# Variables categóricas (texto)
# Revisar si hay columnas de tipo object (texto/categóricas)
categorical_cols = df.select_dtypes(include='object').columns

if len(categorical_cols) > 0:
    print("=== Estadísticas descriptivas de variables categóricas ===")
    print(df.describe(include='object'))
else:
    print("No existen variables categóricas (texto) en el dataset.")

###### f. Identificar cuál es la variable dependiente y cuáles son las variables independientes.
# Definir la variable dependiente (lo que quieres predecir/explicar)
target = "Concrete compressive strength"   #*** aquí poner el nombre de la variable objetivo

# Definir las variables independientes (factores explicativos)
features = [col for col in df.columns if col != target]

print("Variable dependiente (Y):", target)
print("Variables independientes (X):", features)




## Conclusión:

La base de datos está completa, limpia y con variables numéricas, con suficiente variabilidad en las mezclas y resultados. **La resistencia a la compresión del concreto** es la variable objetivo, y las demás son factores explicativos.

# **🔵 Parte 2. Análisis de relaciones entre variables**

    a. Construir una matriz de correlación.

    b. Representar gráficamente las correlaciones mediante un mapa de calor.
    
    c. Identificar las variables que tienen mayor correlación positiva y negativa con la resistencia del concreto.

    d. Identificar si existen variables independientes altamente correlacionadas entre sí.
  
  **Pregunta de análisis: ¿Cuáles variables parecen estar más relacionadas con la resistencia del concreto? ¿Existen posibles problemas de multicolinealidad?**

### **a. Construir una matriz de correlación y b. Representar gráficamente las correlaciones mediante un mapa de calor.**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Construir la matriz de correlación (solo variables numéricas)
correlation_matrix = df.corr(method='pearson')

# 2. Mostrar la matriz en consola
print("=== MATRIZ DE CORRELACIÓN ===")
print(correlation_matrix)

# 3. Visualizar con un mapa de calor
plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", center=0)
plt.title("Matriz de correlación entre variables")
plt.show()


### **Análisis de correlaciones**

* La resistencia del concreto (Concrete compressive strength) muestra su mayor correlación positiva con el Cemento (0.49). Esto confirma que a mayor cantidad de cemento en la mezcla, mayor resistencia se obtiene.

* También se observa una correlación positiva con el Superplasticizer (0.36) y con la Edad (0.33), lo que indica que el uso de aditivos y el tiempo de curado contribuyen significativamente al aumento de la resistencia.

* La mayor correlación negativa se da con el Agua (-0.29), lo cual refleja que un exceso de agua debilita la mezcla, reduciendo la resistencia.

* Otras variables como los agregados (grueso y fino) también presentan correlaciones negativas moderadas, lo que sugiere que su exceso puede afectar la compactación y resistencia final.

In [ ]:
###### c. Identificar las variables que tienen mayor correlación positiva y negativa con la resistencia del concreto.
# df Variable dependinte
target = "Concrete compressive strength"

# Calcular correlaciones de todas las variables con el target
correlations = df.corr()[target].drop(target)

# Ordenar de mayor a menor correlación
sorted_corr = correlations.sort_values(ascending=False)

print("=== Correlaciones con la resistencia del concreto ===")
print(sorted_corr)

# Mostrar la más positiva y la más negativa
print("\nMayor correlación positiva:", sorted_corr.index[0], sorted_corr.iloc[0])
print("Mayor correlación negativa:", sorted_corr.index[-1], sorted_corr.iloc[-1])


## Conclusión:

La resistencia del concreto depende principalmente de tres factores: cantidad de cemento, uso de aditivos y tiempo de curado. El exceso de agua y agregados disminuye la resistencia. En términos prácticos, para lograr un concreto más fuerte se debe optimizar la dosificación de cemento y aditivos, controlar el agua y garantizar un buen tiempo de curado.

In [ ]:
###### d. Identificar si existen variables independientes altamente correlacionadas entre sí.
# df Variable dependinte
target = "Concrete compressive strength"

# 1. Calcular la matriz de correlación
corr_matrix = df.corr(method='pearson')

# 2. Filtrar solo las variables independientes (excluyendo el target)
indep_vars = [col for col in df.columns if col != target]
corr_indep = corr_matrix.loc[indep_vars, indep_vars]

# 3. Identificar pares con correlación alta (ejemplo: |r| > 0.7)
high_corr = []
for col in corr_indep.columns:
    for row in corr_indep.index:
        if col != row:
            if abs(corr_indep.loc[row, col]) > 0.7: ## umbral de referencia
                high_corr.append((row, col, corr_indep.loc[row, col]))

# 4. Muestra resultados
print("=== Variables independientes altamente correlacionadas (|r| > 0.7) ===")
for pair in high_corr:
    print(pair)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. Definir la variable dependiente
target = "Concrete compressive strength"

# 3. Seleccionar solo las variables independientes
indep_vars = [col for col in df.columns if col != target]

# 4. Calcular la matriz de correlación entre las independientes
corr_indep = df[indep_vars].corr(method='pearson')

# 5. Mostrar la matriz.
print("=== MATRIZ DE CORRELACIÓN ENTRE VARIABLES INDEPENDIENTES ===")
print(corr_indep)

# 6. Visualizar con un mapa de calor
plt.figure(figsize=(10,8))
sns.heatmap(corr_indep, annot=True, cmap="coolwarm", center=0)
plt.title("Mapa de calor: Correlaciones entre variables independientes")
plt.show()


## **¿Cuáles variables parecen estar más relacionadas con la resistencia del concreto?**

## **Variables con correlación más alta**

* **Water y Superplasticizer (-0.66)**: correlación negativa fuerte.
Cuando aumenta el aditivo (superplastificante), se reduce la cantidad de agua necesaria en la mezcla.
Es una relación lógica: el aditivo mejora la trabajabilidad del concreto sin necesidad de más agua.

* **Cement y Fly Ash (-0.40)**: correlación moderada negativa.
A mayor uso de cenizas volantes, suele disminuir la cantidad de cemento, porque las cenizas reemplazan parte del cemento.

* **Blast Furnace Slag y Fly Ash (-0.32)**: correlación moderada negativa.
Ambas son materiales cementantes suplementarios, por eso tienden a variar en sentido contrario: si se usa más escoria, se usa menos ceniza.

## **¿Existen posibles problemas de multicolinealidad?**

No se identifican problemas significativos de multicolinealidad entre las variables independientes. Aunque existe una correlación negativa relativamente alta entre Water y Superplasticizer (r = -0.66), ninguna de las correlaciones observadas supera el umbral comúnmente aceptado de ±0.70.

# **🔵 Parte 3. Construcción de modelos**

1.   Elemento de la lista
2.   Elemento de la lista



Dividir los datos utilizando:
* 80% para entrenamiento.
* 20% para prueba.
* random_state = 42.

Construir los siguientes modelos:
* Modelo 1. Regresión Lineal (OLS): Entrenar un modelo mediante mínimos cuadrados ordinarios para predecir la resistencia del concreto.

* Modelo 2. Regresión Ridge:Entrenar un modelo con regularización Ridge utilizando alpha = 1.0.

* Modelo 3. Regresión Lasso: Entrenar un modelo con regularización Lasso utilizando alpha = 1.0.

**Para Ridge y Lasso, realizar el escalamiento de las variables predictoras, ajustándolo exclusivamente con los datos de entrenamiento.**

No es necesario realizar búsqueda de hiperparámetros ni validación cruzada para esta tarea.

In [ ]:
import sklearn
# Cargar librerias

# Tratamiento de Datos

import pandas as pd
import numpy as np

#Graficos

import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns

#Preprocesamiento y Modelo

from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import ElasticNetCV

#Configuracion de nuestro visuales

plt.rcParams['image.cmap'] = 'bwr'
plt.rcParams['savefig.bbox'] = 'tight'
style.use('ggplot') or plt.style.use('ggplot')

#Ignorar las advertencias de paquetes y sus actualizaciones
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from sklearn.model_selection import train_test_split

# Variables independientes (X) y dependiente (y)
X = df.drop(columns = 'Concrete compressive strength')
y = df['Concrete compressive strength']

# Division de los datos
#20% EN EL TEST 43
#80% EN EL TRAIN 172
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y.values.reshape(-1,1),
    test_size = 0.20,
    random_state = 42,
    shuffle = True)

# Verificar dimensiones
print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

### **a. Construir los siguientes modelos:**


#### **Modelo 1. Regresión Lineal (OLS)**

In [ ]:
# Crear y entrenar el modelo

modelo = LinearRegression() ###---Crea el modelo
modelo.fit(X = X_train, y = y_train) ###---Entrena el modelo

# Coeficientes del modelo

df_coeficientes = pd.DataFrame(
    {'predictor':X_train.columns,
     'coef': modelo.coef_.flatten()})

fig, ax = plt.subplots(figsize =(20,5))
ax.stem(df_coeficientes.predictor, df_coeficientes.coef, markerfmt=' ')
plt.xticks(rotation = 90, ha = 'right', size = 9)
ax.set_ylabel('Coeficientes')
ax.set_xlabel('variable')
ax.set_title('Coeficiente del Modelo')
plt.show()

* **Superplasticizer** tiene el coeficiente positivo más alto.
   * Es la variable que más contribuye a aumentar la resistencia del concreto cuando las demás variables permanecen constantes.

* **Age** y **Cement** también tienen coeficientes positivos importantes.
   * A mayor tiempo de curado y mayor cantidad de cemento, mayor resistencia del concreto.

* **Water** presenta el coeficiente negativo más fuerte.
   * Un aumento en la cantidad de agua reduce la resistencia del concreto.

* **Blast Furnace Slag** y **Fly Ash** tienen un efecto positivo, pero menor.

* **Coarse Aggregate** y **Fine Aggregate** tienen coeficientes cercanos a cero.
   * Su impacto sobre la resistencia es relativamente pequeño en comparación con las demás variables.

El modelo indica que las variables con mayor influencia positiva sobre la resistencia del concreto son **Superplasticizer**, **Age** y **Cement**, mientras que **Water** es la variable con mayor efecto negativo

In [ ]:
# Predicciones sobre el conjunto de prueba
predicciones = modelo.predict(X_test)

# Convertir a vector si es necesario
predicciones = predicciones.flatten()

# Mostrar las primeras 10 predicciones
print(predicciones[:10])

In [ ]:
from sklearn.metrics import r2_score

r2_ols = r2_score(
    y_true  = y_test,
    y_pred = predicciones)

print(f'El coeficiente determinacion del test es: {r2_ols}')

### **Conclusión:**
El modelo presenta un coeficiente de determinación (R²) de 0.6275, lo que indica que explica aproximadamente el 62.75% de la variabilidad en la resistencia del concreto. Esto sugiere un desempeño moderadamente bueno, ya que el modelo logra capturar una parte importante del comportamiento de la variable objetivo, aunque aún existe margen de mejora.

#### **Modelo 2. Regresión Ridge**

In [ ]:
X.describe().loc[['min', 'max']]

In [ ]:
# =====================================================
# 1. Estandarización de las variables independientes
# =====================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Ajustar y transformar entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Transformar prueba
X_test_scaled = scaler.transform(X_test)


# =====================================================
# 2. Crear y entrenar el modelo Ridge
# =====================================================

from sklearn.linear_model import Ridge

modelo_ridge = Ridge(
    alpha=1.0,
    fit_intercept=True
)

modelo_ridge.fit(X_train_scaled, y_train)


# =====================================================
# 3. Realizar predicciones
# =====================================================

predicciones_ridge = modelo_ridge.predict(X_test_scaled)


# =====================================================
# 4. Evaluar el modelo
# =====================================================

r2_ridge = modelo_ridge.score(X_test_scaled, y_test)

print("Coeficiente de determinación (R²):", r2_ridge)


# =====================================================
# 5. Mostrar algunas predicciones
# =====================================================

print("\nPrimeras 10 predicciones:")
print(predicciones_ridge[:10])

In [ ]:
from sklearn.linear_model import Ridge
import numpy as np
import matplotlib.pyplot as plt

# Rango de alphas para visualizar el comportamiento
alphas = np.logspace(-5, 5, 200)

coefs = []

for alpha in alphas:
    modelo_temp = Ridge(
        alpha=alpha,
        fit_intercept=True
    )

    modelo_temp.fit(X_train_scaled, y_train)
    coefs.append(modelo_temp.coef_)

# Gráfico
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(alphas, coefs)
ax.set_xscale('log')

ax.set_title('Evolución de los coeficientes según α')
ax.set_xlabel('Alpha (escala logarítmica)')
ax.set_ylabel('Coeficientes')

plt.axis('tight')
plt.show()

### **Conclusión**:

* Cuando **α** es pequeño, los coeficientes tienen valores más altos y el modelo se comporta de manera similar a una regresión lineal tradicional.

* A medida que *α* aumenta, los coeficientes se reducen progresivamente y se acercan a cero.

* Ningún coeficiente desaparece completamente, lo cual es una característica propia de Ridge: reduce la magnitud de los coeficientes, pero no elimina variables.

La gráfica muestra que el modelo Ridge aplica correctamente la regularización, reduciendo gradualmente la magnitud de los coeficientes a medida que aumenta el valor de α. Esto ayuda a disminuir la complejidad del modelo y a mejorar su capacidad de generalización, evitando que variables con mayor influencia dominen excesivamente las predicciones.

#### **Modelo 3. Regresión Lasso**

In [ ]:
from sklearn.linear_model import Lasso

# Modelo Lasso con alpha fijo
modelo_lasso = Lasso(
    alpha=1.0,
    fit_intercept=True,
    max_iter=10000
)

# Entrenamiento del modelo
modelo_lasso.fit(X_train_scaled, y_train)

In [ ]:
# Predicciones
predicciones_lasso = modelo_lasso.predict(X_test_scaled)

# R²
r2_lasso = modelo_lasso.score(X_test_scaled, y_test)

print("Coeficiente de determinación (R²):", r2_lasso)

El modelo Lasso obtuvo un coeficiente de determinación (R²) de 0.5639, indicando que explica el 56.39% de la variación de la resistencia del concreto. Aunque su capacidad predictiva es inferior a la de la regresión lineal y Ridge, Lasso ofrece la ventaja de realizar selección automática de variables, produciendo un modelo más simple y fácil de interpretar al eliminar aquellas características con menor contribución al resultado.

In [ ]:
from sklearn.linear_model import Lasso
import numpy as np
import matplotlib.pyplot as plt

# Rango de alphas para visualizar el comportamiento
alphas = np.logspace(-5, 2, 200)

coefs = []

for alpha in alphas:

    modelo_temp = Lasso(
        alpha=alpha,
        fit_intercept=True,
        max_iter=10000
    )

    modelo_temp.fit(X_train_scaled, y_train)
    coefs.append(modelo_temp.coef_)

# Gráfica
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(alphas, coefs)

# Línea vertical para alpha = 1.0
ax.axvline(
    x=1.0,
    color='red',
    linestyle='--',
    label='Alpha = 1.0'
)

ax.set_xscale('log')
ax.set_ylabel('Coeficientes')
ax.set_xlabel('Alpha')
ax.set_title('Coeficientes del modelo Lasso en función de la regularización')

ax.legend()
plt.axis('tight')
plt.show()

La gráfica evidencia que, a medida que aumenta el parámetro de regularización (α), varios coeficientes son reducidos exactamente a cero, confirmando la capacidad de Lasso para realizar selección automática de variables. Esto indica que no todas las variables aportan la misma información para explicar la resistencia del concreto y que el modelo identifica un subconjunto de predictores más relevantes. Como resultado, se obtiene un modelo más parsimonioso e interpretable, aunque con una ligera pérdida de capacidad predictiva respecto a modelos menos restrictivos como Ridge.

## Brebe validación de las variables que elimina y conserva lasso.

In [ ]:
import pandas as pd

# Crear tabla de coeficientes
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': modelo_lasso.coef_
})

# Ordenar por importancia
coeficientes = coeficientes.sort_values(
    by='Coeficiente',
    key=abs,
    ascending=False
)

print(coeficientes)

### Variables seleccionadas

In [ ]:
variables_seleccionadas = coeficientes[
    coeficientes['Coeficiente'] != 0
]

print("Variables seleccionadas por Lasso:")
print(variables_seleccionadas)

### Variables eliminadas

In [ ]:
variables_eliminadas = coeficientes[
    coeficientes['Coeficiente'] == 0
]

print("Variables eliminadas por Lasso:")
print(variables_eliminadas)

Las variables con coeficientes distintos de cero constituyen el subconjunto de predictores más relevantes identificado por Lasso. Estas variables son las que aportan mayor capacidad explicativa sobre la resistencia del concreto, mientras que las variables con coeficientes nulos fueron descartadas por el proceso de regularización debido a su baja contribución al modelo.

In [ ]:
import pandas as pd

# Crear DataFrame de coeficientes
df_coeficientes = pd.DataFrame({
    'predictor': X.columns,
    'coef': modelo_lasso.coef_
})

# Ordenar por importancia
df_coeficientes = df_coeficientes.sort_values(
    by='coef',
    key=abs,
    ascending=False
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 4))

ax.stem(
    df_coeficientes['predictor'],
    df_coeficientes['coef'],
    markerfmt=' '
)

plt.xticks(
    rotation=90,
    ha='right',
    size=8
)

ax.set_ylabel('Coeficientes')
ax.set_xlabel('Variables')
ax.set_title('Coeficientes del modelo Lasso (α = 1.0)')

plt.tight_layout()
plt.show()

La gráfica muestra que el modelo Lasso (α = 1.0) asigna diferente importancia a las variables predictoras, conservando únicamente aquellas con mayor capacidad explicativa sobre la resistencia del concreto. Las variables con coeficientes más alejados de cero tienen una influencia más significativa en la predicción, mientras que las variables con coeficientes nulos o cercanos a cero aportan poca información y pueden considerarse prescindibles.

# **🔵 Parte 4. Evaluación de modelos**:

Para los tres modelos desarrollados:

* Realizar predicciones sobre los datos de prueba.
* Calcular el coeficiente de determinación (R²).
* Calcular la raíz del error cuadrático medio (RMSE).
* Presentar una tabla comparativa con los resultados obtenidos.
* Identificar cuál modelo presenta el mejor rendimiento predictivo.

### **a. Realizar predicciones sobre los datos de prueba**

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# =====================
# OLS
# =====================

pred_ols = modelo.predict(X_test)

r2_ols = r2_score(y_test, pred_ols)
rmse_ols = np.sqrt(mean_squared_error(y_test, pred_ols))

print("OLS")
print("R²:", r2_ols)
print("RMSE:", rmse_ols)

# =====================
# Ridge
# =====================

pred_ridge = modelo_ridge.predict(X_test_scaled)

r2_ridge = r2_score(y_test, pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))

print("\nRidge")
print("R²:", r2_ridge)
print("RMSE:", rmse_ridge)

# =====================
# Lasso
# =====================

pred_lasso = modelo_lasso.predict(X_test_scaled)

r2_lasso = r2_score(y_test, pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, pred_lasso))

print("\nLasso")
print("R²:", r2_lasso)
print("RMSE:", rmse_lasso)

In [ ]:
display(tabla_resultados.style.format({
    'R2': '{:.4f}',
    'RMSE': '{:.4f}'
}))

In [ ]:
mejor_modelo = tabla_resultados.iloc[0]

print("Mejor modelo:")
print("Modelo:", mejor_modelo['Modelo'])
print("R²:", round(mejor_modelo['R2'], 4))
print("RMSE:", round(mejor_modelo['RMSE'], 4))

## **Conclusión:**

Después de explorar los datos, analizar las correlaciones, verificar la ausencia de problemas graves de multicolinealidad y construir los modelos OLS, Ridge y Lasso, se realizó una comparación objetiva utilizando dos criterios: capacidad de explicación (R²) y error de predicción (RMSE).

Los resultados muestran que el modelo Ridge obtuvo el mejor desempeño global, alcanzando un R² = 0.6276 y un RMSE = 9.7962, valores ligeramente mejores que los obtenidos por OLS (R² = 0.6276; RMSE = 9.7965). Por su parte, Lasso presentó un menor rendimiento predictivo (R² = 0.5639; RMSE = 10.6004).

Aunque las diferencias entre OLS y Ridge son pequeñas, Ridge ofrece una ventaja importante: controla mejor la influencia de las variables mediante una regularización que evita que pequeñas variaciones en los datos afecten excesivamente al modelo. Esto lo convierte en una alternativa más estable y robusta para realizar predicciones futuras. La evolución de los coeficientes muestra precisamente cómo Ridge reduce gradualmente la influencia de las variables sin eliminarlas, conservando toda la información disponible.

En contraste, Lasso eliminó algunas variables al llevar sus coeficientes a cero, simplificando el modelo, pero sacrificando capacidad predictiva. Esto explica por qué obtuvo un R² menor y un RMSE más alto.


# **🔵 Parte 5. Conclusiones**


Responder con sus propias palabras:

¿**Cuál de los tres modelos obtuvo el mejor resultado y por qué?**
* El modelo Ridge fue seleccionado como el mejor modelo para predecir la resistencia del concreto porque presentó el mayor nivel de precisión y el menor error de predicción. Además, mantiene todas las variables relevantes del problema y genera un modelo más estable.

**¿Qué significa el valor de R² obtenido por el mejor modelo?**
* El R2 de Ridge indica que puede explicar aproximadamente el 63% del comportamiento de la resistencia utilizando la información de los materiales y la edad del concreto. Este modelo fue el que cometió menos errores.

**¿Qué representa el RMSE en el contexto de la resistencia del concreto?**
* Es la predicción, es el promedio cuando intenta predecir la resitencia del concreto. Cuando el modelo predice la resistencia del concreto, normalmente se equivoca alrededor de 9.8 unidades respecto al valor real.

**¿Qué diferencias encontró entre OLS, Ridge y Lasso?**
* OLS untilizo toda la información disponible y asigno un peso a cada variable y no palica ningún mecanismo para controlar la influencia de alguna variable.  Por el contrario, **Ridge** si reduce legeramente la influencia de las variables más fuertes con el fin de evitar que el modelo dependa de pocas características. Ahora, **Lasso** elimina algunas variables que considera poco relevantes.

**¿Considera que estos modelos podrían resultar útiles para la empresa?**
* Desde mi rol como analista de datos, si considero que estos modelos son relevantes para ser aplicados en mi trabajo. Facilitarían y optimizarían los proceso de predicción de provisiones para las carteras del Banco a corte diario.